In [2]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from tools import DuckDuckGoSearchRun, weather_info_tool, hub_stats_tool
from retriever import guest_info_tool

In [ ]:
search_tool = DuckDuckGoSearchRun()

llm = HuggingFaceEndpoint(
    # repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    repo_id="Qwen/Qwen3.8-27B:novita",
)

chat = ChatHuggingFace(llm=llm, verbose=True)
tools = [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

### Ex 1: Finding guest info

In [4]:
response = alfred.invoke({"messages": "Tell me about 'Lady Ada Lovelace'"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Lady Ada Lovelace is described as your best friend and an esteemed mathematician. She's renowned for her pioneering work in mathematics and computing, and is often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.

Her contact email is ada.lovelace@example.com.


### Ex 2: Checking the Weather for Fireworks

In [5]:
response = alfred.invoke({"messages": "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Great news for your event! Tonight in Paris it's **clear and 25°C (77°F)** — ideal conditions for a fireworks display.

Why it works well:
- **Clear skies** mean excellent visibility for the fireworks — no clouds to obscure the display.
- **25°C** is comfortable for guests watching outdoors.
- No rain or wind warnings reported, which reduces launch and safety risks.

Just a couple of practical tips:
- Even with clear conditions, double-check wind speed closer to launch time, as gusts can affect shell trajectories.
- Make sure your local permits and safety zones are in place — Paris has specific regulations for public fireworks displays.

Looking like a beautiful evening for it! 🎆


### Ex 3: Impressing AI Researchers

In [6]:
response = alfred.invoke({"messages": "One of our guests is from Qwen. What can you tell me about their most popular model?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Your guest from Qwen has a very popular model on the Hugging Face Hub: **Qwen/Qwen3-0.6B**, with over **24.1 million downloads** — the most downloaded model from the Qwen team. It's a compact, efficient small language model from their Qwen3 series, well-suited for lightweight or on-device deployments.


### Ex 4: Combining Multiple Tools

In [7]:
response = alfred.invoke({"messages":"I need to speak with 'Dr. Nikola Tesla' about recent advancements in wireless energy. Can you help me prepare for this conversation?"})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Great, I found your contact info and pulled together some current research context. Here's your prep sheet:

## About Your Guest
- **Dr. Nikola Tesla** — an old friend from your university days
- He has **recently patented a new wireless energy transmission system** and is eager to discuss it with you
- **Small talk tip:** He's passionate about pigeons — a perfect icebreaker to warm things up
- **Contact:** nikola.tesla@gmail.com

## Recent Advancements in Wireless Energy (good topics to raise)
1. **High-efficiency rectennas & metamaterials** — new designs that significantly improve how much energy receivers can capture from the air
2. **Adaptive impedance matching circuits** — dynamic tuning that boosts transfer efficiency, especially as distance or receiver position changes
3. **Commercial WPT expansion** — wireless power transfer is now available for EVs, drones, and consumer electronics, though **fragmented standards and high infrastructure costs** remain barri

##  Advanced Features: Conversation Memory

In [8]:
# First interaction
response = alfred.invoke({"messages": [HumanMessage(content="Tell me about 'Lady Ada Lovelace'. What's her background and how is she related to me?")]})


print("🎩 Alfred's Response:")
print(response['messages'][-1].content)
print()

# Second interaction (referencing the first)
response = alfred.invoke({"messages": response["messages"] + [HumanMessage(content="What projects is she currently working on?")]})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Lady Ada Lovelace** is your **best friend**!

### Background
She is an esteemed mathematician, renowned for her pioneering work in mathematics and computing. She is widely celebrated as the **first computer programmer** due to her groundbreaking work on **Charles Babbage's Analytical Engine** — a hypothetical mechanical general-purpose computer. Her notes on the Engine included what is considered the first algorithm intended to be carried out by a machine, making her a true pioneer in the field of computer science.

### Your Relationship
Beyond her historical significance, she holds a special place in your personal life as your **best friend**.

If you'd like to reach out to her, her email is **ada.lovelace@example.com**.

🎩 Alfred's Response:
I don't have any information about Ada's current projects. The guest records only contain her background (mathematician, pioneer of computing, work on Babbage's Analytical Engine) and her contact details — no details on wha